
# Pre-processamento de Dados para o Milestone 2

Grupo 2

João Costa | 113564

Sílvia Ribeiro | 129428

Vitória Rodrigues | 130557

Este notebook realiza o pre-processamento do dataset integrado produzido no notebook `01_integracaoDados.ipynb`, de acordo com os requisitos definidos para o **Milestone 2**:

- tratamento de valores em falta
- identificacao e tratamento de outliers
- transformacoes aplicadas
  - normalizacao
  - logaritmos
- criacao de variaveis derivadas e respetiva justificacao

O objetivo e preparar duas versoes do dataset:

- `df_preprocessed_eda`: versao limpa e interpretavel, adequada para analise exploratoria
- `df_model_ready`: versao mais estruturada e numerica, adequada para etapas posteriores de modelacao


In [ ]:

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)



## 1. Carregamento dos datasets


In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

START_DIR = Path.cwd().resolve()
print(f"Diretoria atual de execução: {START_DIR}")

DATA_DIR = None
for root in [START_DIR, *START_DIR.parents]:
    candidate = root / "dataset"
    if candidate.exists() and candidate.is_dir():
        DATA_DIR = candidate.resolve()
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Não foi possível localizar a pasta central 'dataset'. "
        "Certifica-te de que manténs a pasta 'dataset' na raiz do projeto."
    )

print(f"Pasta unificada de dados encontrada em: {DATA_DIR}")

INTEGRATED_DATA_PATH = DATA_DIR / "df_final_integrado_multi_poi_com_meteorologia.csv"

if not INTEGRATED_DATA_PATH.exists():
    raise FileNotFoundError(
        f"O ficheiro integrado não foi encontrado: {INTEGRATED_DATA_PATH}. "
        "Certifica-te de que executaste o Notebook 1 com sucesso antes de rodar este código."
    )

df_raw = pd.read_csv(INTEGRATED_DATA_PATH, low_memory=False)
print(f"Dataset integrado carregado com sucesso! Dimensões: {df_raw.shape}")
display(df_raw.head())


## 2. Auditoria inicial da qualidade dos dados

Antes de aplicar transformacoes, importa compreender:

- cobertura temporal e numero de POIs presentes
- colunas totalmente vazias
- colunas com valores em falta
- distribuicao inicial das variaveis numericas


In [ ]:

def inspect_dataframe(df, name, head_rows=5):
    print(f"{name} - shape: {df.shape}")
    display(df.head(head_rows))
    df.info()
    print("\nMissing values (%):")
    display((df.isna().mean().sort_values(ascending=False) * 100).round(2).to_frame("missing_pct"))

inspect_dataframe(df_raw, "Dataset integrado bruto")

print("Numero de POIs:", df_raw["geography_key"].nunique())
print("POIs presentes:", df_raw["poi_name"].dropna().unique().tolist())
print("Cobertura temporal:", df_raw["movement_date"].min(), "a", df_raw["movement_date"].max())


In [ ]:

missing_ratio = df_raw.isna().mean().sort_values(ascending=False)
all_missing_columns = missing_ratio[missing_ratio.eq(1.0)].index.tolist()
partially_missing_columns = missing_ratio[(missing_ratio.gt(0.0)) & (missing_ratio.lt(1.0))].index.tolist()

print("Colunas totalmente vazias:", all_missing_columns)
print("Colunas parcialmente incompletas:", partially_missing_columns)

numeric_summary = df_raw.select_dtypes(include=["number", "bool"]).describe().T
numeric_summary = numeric_summary[["mean", "std", "min", "25%", "50%", "75%", "max"]]
display(numeric_summary)



## 3. Estrategia de pre-processamento

A estrategia adotada neste notebook e conservadora e transparente:

1. Corrigir tipos de dados e ordenar a serie temporal por POI.
2. Tratar valores em falta sem eliminar observacoes, dado que o dataset e pequeno.
3. Identificar outliers por IQR nas variaveis numericas continuas.
4. Tratar outliers por **clipping** (winsorizacao conservadora), evitando remover linhas.
5. Aplicar transformacoes pedidas no milestone:
   - normalizacao min-max
   - transformacoes logaritmicas com `log1p`
6. Criar variaveis derivadas com utilidade clara para EDA e modelacao posterior.


In [ ]:

df = df_raw.copy()

df["movement_date"] = pd.to_datetime(df["movement_date"], errors="coerce")
df = df.sort_values(["geography_key", "movement_date"]).reset_index(drop=True)

print(df[["movement_date", "geography_key", "poi_name"]].head(10).to_string(index=False))



## 4. Tratamento de valores em falta

### 4.1. Colunas totalmente vazias

As colunas completamente vazias nao acrescentam informacao ao dataset e sao removidas nesta fase.


In [ ]:

columns_to_drop = [
    "island_name",
    "concelho_name",
    "freguesia_name",
    "latitude",
    "longitude",
]

columns_to_drop = [column for column in columns_to_drop if column in df.columns]
df = df.drop(columns=columns_to_drop)

print("Colunas removidas por estarem totalmente vazias:", columns_to_drop)
print("Shape apos remocao:", df.shape)



### 4.2. Reconstituir variaveis temporais a partir de `movement_date`

Algumas observacoes apresentam valores em falta nas colunas provenientes da `DIM_DATE`. Como a data existe na coluna `movement_date`, essas variaveis sao reconstruidas diretamente a partir do calendario.


In [ ]:

pt_day_names = {
    1: "Segunda",
    2: "Ter?a",
    3: "Quarta",
    4: "Quinta",
    5: "Sexta",
    6: "S?bado",
    7: "Domingo",
}

pt_month_names = {
    1: "Janeiro",
    2: "Fevereiro",
    3: "Mar?o",
    4: "Abril",
    5: "Maio",
    6: "Junho",
    7: "Julho",
    8: "Agosto",
    9: "Setembro",
    10: "Outubro",
    11: "Novembro",
    12: "Dezembro",
}

def fill_from_date(df_input: pd.DataFrame) -> pd.DataFrame:
    df_out = df_input.copy()
    dt = df_out["movement_date"]

    weekday_num = dt.dt.dayofweek + 1
    iso_week = dt.dt.isocalendar().week.astype("Int64")

    df_out["date_value"] = df_out["date_value"].fillna(dt.dt.strftime("%Y-%m-%d"))
    df_out["day_of_week"] = df_out["day_of_week"].fillna(weekday_num)
    df_out["day_name"] = df_out["day_name"].fillna(weekday_num.map(pt_day_names))
    df_out["day_of_month"] = df_out["day_of_month"].fillna(dt.dt.day)
    df_out["day_of_year"] = df_out["day_of_year"].fillna(dt.dt.dayofyear)
    df_out["week_of_year"] = df_out["week_of_year"].fillna(iso_week)
    df_out["month_number"] = df_out["month_number"].fillna(dt.dt.month)
    df_out["month_name"] = df_out["month_name"].fillna(dt.dt.month.map(pt_month_names))
    df_out["quarter_number"] = df_out["quarter_number"].fillna(dt.dt.quarter)
    df_out["year_number"] = df_out["year_number"].fillna(dt.dt.year)

    season_by_month = (
        df_out.loc[df_out["season"].notna(), ["month_number", "season"]]
        .drop_duplicates()
        .groupby("month_number")["season"]
        .agg(lambda x: x.mode().iloc[0])
        .to_dict()
    )
    peak_season_by_month = (
        df_out.loc[df_out["is_peak_season"].notna(), ["month_number", "is_peak_season"]]
        .drop_duplicates()
        .groupby("month_number")["is_peak_season"]
        .agg(lambda x: x.mode().iloc[0])
        .to_dict()
    )

    df_out["season"] = df_out.apply(
        lambda row: row["season"] if pd.notna(row["season"]) else season_by_month.get(row["month_number"], "Shoulder"),
        axis=1,
    )
    df_out["is_peak_season"] = df_out.apply(
        lambda row: row["is_peak_season"] if pd.notna(row["is_peak_season"]) else peak_season_by_month.get(row["month_number"], False),
        axis=1,
    )

    return df_out

df = fill_from_date(df)



### 4.3. Tratar variaveis booleanas e feriados

As variaveis booleanas sao convertidas para um formato consistente e os valores em falta sao preenchidos de forma justificada:

- `is_weekend`: derivado da data
- `is_holiday`: preenchido com `False` quando ausente
- `is_peak_season`: preenchido com base no mes, a partir do padrao existente
- `is_business_day`: recalculado a partir de `is_weekend` e `is_holiday`
- `holiday_name`: preenchido com `Nao_feriado` quando aplicavel


In [ ]:

def parse_boolean_column(series: pd.Series) -> pd.Series:
    if str(series.dtype) == "bool":
        return series.astype("boolean")

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
    }
    normalized = series.astype("string").str.strip().str.lower()
    return normalized.map(mapping).astype("boolean")

for column in ["is_weekend", "is_holiday", "is_peak_season", "is_business_day", "is_sensorized"]:
    df[column] = parse_boolean_column(df[column])

df["is_weekend"] = df["is_weekend"].fillna(df["movement_date"].dt.dayofweek >= 5)
df["is_holiday"] = df["is_holiday"].fillna(False)
df["is_peak_season"] = df["is_peak_season"].fillna(False)
df["is_business_day"] = (~df["is_weekend"] & ~df["is_holiday"]).astype("boolean")
df["holiday_name"] = df["holiday_name"].fillna("Nao_feriado")

missing_after_imputation = df.isna().mean().sort_values(ascending=False)
display((missing_after_imputation * 100).round(2).to_frame("missing_pct_after_imputation"))



## 5. Identificacao e tratamento de outliers

Os outliers sao identificados com base no metodo do **IQR (Interquartile Range)**, aplicado a variaveis numericas continuas.

Dado o reduzido numero de observacoes, os outliers **nao sao removidos**. Em vez disso, e aplicada uma estrategia de clipping conservador (winsorizacao), preservando todas as linhas do dataset.


In [ ]:

continuous_columns = [
    "incoming_record_count",
    "weather_temperature_2m_mean",
    "weather_temperature_2m_min",
    "weather_temperature_2m_max",
    "weather_precipitation_sum",
    "weather_wind_speed_10m_mean",
    "weather_wind_speed_10m_max",
    "weather_relative_humidity_2m_mean",
]

outlier_summary = []

def iqr_bounds(series: pd.Series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return q1, q3, lower, upper

for column in continuous_columns + ["total_incoming"]:
    q1, q3, lower, upper = iqr_bounds(df[column])
    flag_column = f"outlier_flag_{column}"
    df[flag_column] = ((df[column] < lower) | (df[column] > upper))
    outlier_summary.append(
        {
            "column": column,
            "q1": q1,
            "q3": q3,
            "lower_bound": lower,
            "upper_bound": upper,
            "outlier_count": int(df[flag_column].sum()),
        }
    )

outlier_summary_df = pd.DataFrame(outlier_summary).sort_values("outlier_count", ascending=False)
display(outlier_summary_df)


In [ ]:

df_preprocessed = df.copy()

# O target original e preservado. O clipping e aplicado apenas a variaveis explicativas continuas.
for column in continuous_columns:
    _, _, lower, upper = iqr_bounds(df_preprocessed[column])
    df_preprocessed[column] = df_preprocessed[column].clip(lower=lower, upper=upper)

comparison_columns = continuous_columns[:4]
comparison = pd.DataFrame({f"original_{col}": df[col] for col in comparison_columns})
for col in comparison_columns:
    comparison[f"clipped_{col}"] = df_preprocessed[col]

display(comparison.head(10))



## 6. Transformacoes aplicadas

### 6.1. Transformacoes logaritmicas

A transformacao `log1p` e aplicada a variaveis assimetricas e nao negativas, com o objetivo de reduzir a influencia de valores extremos e estabilizar a escala.

Variaveis transformadas:

- `total_incoming`
- `incoming_record_count`
- `weather_precipitation_sum`


In [ ]:

log_columns = [
    "total_incoming",
    "incoming_record_count",
    "weather_precipitation_sum",
]

for column in log_columns:
    df_preprocessed[f"log_{column}"] = np.log1p(df_preprocessed[column])

log_preview_columns = log_columns + [f"log_{column}" for column in log_columns]
display(df_preprocessed[log_preview_columns].head(10))



### 6.2. Normalizacao min-max

A normalizacao min-max e aplicada a variaveis numericas explicativas continuas, colocando-as numa escala comum entre 0 e 1.

A variavel alvo original nao e substituida nesta etapa, para preservar interpretabilidade no dataset destinado a EDA.


In [ ]:

normalization_columns = [
    "incoming_record_count",
    "incoming_source_rows",
    "day_of_month",
    "day_of_year",
    "week_of_year",
    "month_number",
    "quarter_number",
    "weather_hourly_records",
    "weather_temperature_2m_mean",
    "weather_temperature_2m_min",
    "weather_temperature_2m_max",
    "weather_precipitation_sum",
    "weather_wind_speed_10m_mean",
    "weather_wind_speed_10m_max",
    "weather_relative_humidity_2m_mean",
]

for column in normalization_columns:
    col_min = df_preprocessed[column].min()
    col_max = df_preprocessed[column].max()
    if pd.isna(col_min) or pd.isna(col_max) or col_min == col_max:
        df_preprocessed[f"norm_{column}"] = 0.0
    else:
        df_preprocessed[f"norm_{column}"] = (df_preprocessed[column] - col_min) / (col_max - col_min)

norm_preview = [column for column in df_preprocessed.columns if column.startswith("norm_")][:10]
display(df_preprocessed[norm_preview].head(10))



## 7. Criacao de variaveis derivadas e justificacao

As variaveis derivadas abaixo foram criadas para enriquecer o dataset de forma interpretavel e potencialmente util para modelacao posterior.

### Variaveis criadas

- `weather_temperature_range`: amplitude termica diaria
- `has_precipitation`: indicador de dias com precipitacao
- `is_non_business_day`: indicador de fins de semana ou feriados
- `day_of_week_sin` e `day_of_week_cos`: codificacao ciclica do dia da semana
- `month_sin` e `month_cos`: codificacao ciclica do mes
- `days_since_first_observation`: numero de dias decorridos desde a primeira observacao de cada POI

### Justificacao

- A amplitude termica ajuda a capturar variabilidade intra-diaria das condicoes meteorologicas.
- O indicador de precipitacao simplifica a interpretacao de dias com chuva.
- O indicador de nao-dia-util aproxima efeitos de lazer e turismo.
- As codificacoes ciclicas evitam descontinuidades artificiais nas variaveis temporais.
- A variavel de dias decorridos por POI ajuda a representar a progressao temporal dentro de cada serie.


In [ ]:

df_preprocessed["weather_temperature_range"] = (
    df_preprocessed["weather_temperature_2m_max"] - df_preprocessed["weather_temperature_2m_min"]
)
df_preprocessed["has_precipitation"] = df_preprocessed["weather_precipitation_sum"] > 0
df_preprocessed["is_non_business_day"] = df_preprocessed["is_weekend"] | df_preprocessed["is_holiday"]

df_preprocessed["day_of_week_sin"] = np.sin(2 * np.pi * df_preprocessed["day_of_week"] / 7)
df_preprocessed["day_of_week_cos"] = np.cos(2 * np.pi * df_preprocessed["day_of_week"] / 7)
df_preprocessed["month_sin"] = np.sin(2 * np.pi * df_preprocessed["month_number"] / 12)
df_preprocessed["month_cos"] = np.cos(2 * np.pi * df_preprocessed["month_number"] / 12)

df_preprocessed["days_since_first_observation"] = (
    df_preprocessed.groupby("geography_key")["movement_date"]
    .transform(lambda series: (series - series.min()).dt.days)
)

derived_columns = [
    "weather_temperature_range",
    "has_precipitation",
    "is_non_business_day",
    "day_of_week_sin",
    "day_of_week_cos",
    "month_sin",
    "month_cos",
    "days_since_first_observation",
]
display(df_preprocessed[["movement_date", "poi_name"] + derived_columns].head(10))



## 8. Preparar a versao para EDA e a versao model-ready

A versao `df_preprocessed_eda` preserva legibilidade e interpretabilidade.

A versao `df_model_ready` converte booleans para inteiros, aplica one-hot encoding nas categorias relevantes e mantem apenas colunas adequadas para modelacao posterior, sem perder as variaveis alvo e de referencia mais importantes.


In [ ]:

df_preprocessed_eda = df_preprocessed.copy()

for column in ["is_weekend", "is_holiday", "is_peak_season", "is_business_day", "is_sensorized", "has_precipitation", "is_non_business_day"]:
    df_preprocessed_eda[column] = df_preprocessed_eda[column].astype("boolean")

print("Shape df_preprocessed_eda:", df_preprocessed_eda.shape)
display(df_preprocessed_eda.head())


In [ ]:

df_model_ready = df_preprocessed_eda.copy()

boolean_columns = [column for column in df_model_ready.columns if str(df_model_ready[column].dtype) == "boolean"]
for column in boolean_columns:
    df_model_ready[column] = df_model_ready[column].astype(int)

categorical_columns = [
    "poi_name",
    "poi_category",
    "season",
]

# Colunas textuais redundantes ou apenas descritivas para referencia humana
columns_to_remove_from_model = [
    "movement_date",
    "date_value",
    "day_name",
    "month_name",
    "holiday_name",
    "geography_name",
    "geography_type",
    "timezone",
]

columns_to_remove_from_model = [column for column in columns_to_remove_from_model if column in df_model_ready.columns]
df_model_ready = df_model_ready.drop(columns=columns_to_remove_from_model)

df_model_ready = pd.get_dummies(
    df_model_ready,
    columns=[column for column in categorical_columns if column in df_model_ready.columns],
    drop_first=False,
    dtype=int,
)

print("Shape df_model_ready:", df_model_ready.shape)
display(df_model_ready.head())



## 9. Validacao final do pre-processamento

Nesta etapa final, sao verificadas:

- dimensao final das duas versoes exportadas
- ausencia de colunas totalmente vazias
- valores em falta remanescentes


In [ ]:

print("df_preprocessed_eda shape:", df_preprocessed_eda.shape)
print("df_model_ready shape:", df_model_ready.shape)

print("\nMissing values no df_preprocessed_eda (%):")
display((df_preprocessed_eda.isna().mean().sort_values(ascending=False) * 100).round(2).to_frame("missing_pct"))

print("\nMissing values no df_model_ready (%):")
display((df_model_ready.isna().mean().sort_values(ascending=False) * 100).round(2).to_frame("missing_pct"))



## 10. Exportar os datasets finais

Sao exportadas duas versoes finais:

- `df_preprocessed_eda.csv`: versao limpa, legivel e adequada para EDA
- `df_model_ready.csv`: versao preparada para etapas posteriores de modelacao


In [ ]:
preprocessed_path = DATA_DIR / "df_preprocessed_eda.csv"
model_ready_path = DATA_DIR / "df_model_ready.csv"

df_preprocessed_eda.to_csv(preprocessed_path, index=False)
df_model_ready.to_csv(model_ready_path, index=False)

print("Datasets pré-processados exportados com sucesso para a pasta central:")
print(f"-> Versão para Análise (EDA): {preprocessed_path} (Dimensões: {df_preprocessed_eda.shape})")
print(f"-> Versão para Modelos (Numérica): {model_ready_path} (Dimensões: {df_model_ready.shape})")


## 11. Observacoes finais

- O tratamento de valores em falta foi realizado de forma conservadora, privilegiando reconstrucao a partir da data e remocao apenas de colunas totalmente vazias.
- O tratamento de outliers foi efetuado por clipping, evitando a eliminacao de observacoes num dataset reduzido.
- As transformacoes logaritmicas e de normalizacao foram mantidas de forma explicita no dataset, para documentar claramente o pre-processamento aplicado.
- As variaveis derivadas foram criadas com justificacao funcional, visando enriquecer o dataset para EDA e para modelacao futura.
